# Private CayleyPy graph input acceptance
16 graph definitions, native 1/2 GPU paths, selected forced LibTorch, CayleyPy backend and explicit rejection checks.


In [ ]:
# Основные параметры
GRAPH_SIZE = 4                    # пример: LRX(4); замените граф в ячейке запуска
START_STATE = [3, 2, 1, 0]
BEAM_WIDTH = 4096
NUM_GPUS = 2                      # ровно столько GPU; нехватка вызывает ошибку
MAX_STEPS = 16
BACKEND = "native"                # "native" или "torch"
WHEEL_PATH = None                 # авто поиск в /kaggle/input; либо полный путь


In [ ]:
from pathlib import Path
import hashlib, subprocess, sys

EXPECTED_SHA256 = "9d91f4e0c66ee927d5343726f6c34574529656a0a73b6a8eb093cfd535ebbd13"
if WHEEL_PATH:
    wheel = Path(WHEEL_PATH)
else:
    candidates = list(Path("/kaggle/input").rglob("cayleypy_native-0.1.0.dev0-py3-none-any.whl"))
    candidates += list(Path("/kaggle/working").glob("cayleypy_native-0.1.0.dev0-py3-none-any.whl"))
    matching = [p for p in candidates if hashlib.sha256(p.read_bytes()).hexdigest() == EXPECTED_SHA256]
    if not matching:
        raise FileNotFoundError("Прикрепите wheel из комплекта как Kaggle input или задайте WHEEL_PATH.")
    wheel = sorted(matching)[0]
assert hashlib.sha256(wheel.read_bytes()).hexdigest() == EXPECTED_SHA256, "Неверная версия wheel"
subprocess.run([sys.executable, "-m", "pip", "install", str(wheel)], check=True)


In [ ]:
import shutil, torch

if not torch.cuda.is_available():
    raise RuntimeError("Включите GPU в настройках ноутбука.")
if type(NUM_GPUS) is not int or not 1 <= NUM_GPUS <= torch.cuda.device_count():
    raise ValueError(f"Запрошено {NUM_GPUS}, доступно {torch.cuda.device_count()} GPU")
if BACKEND not in ("native", "torch"):
    raise ValueError("BACKEND должен быть native или torch")
if BACKEND == "native":
    for tool in ("cmake", "ninja", "c++"):
        if shutil.which(tool) is None:
            raise RuntimeError(f"Не найден инструмент сборки: {tool}")
    if shutil.which("nvcc") is None and not Path("/usr/local/cuda/bin/nvcc").exists():
        raise RuntimeError("Не найден CUDA toolkit/nvcc")
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
for i in range(NUM_GPUS):
    print(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i))
print("Свободно /tmp, GiB:", round(shutil.disk_usage("/tmp").free / 2**30, 2))
CACHE = Path("/tmp/cayleypy-native-cache")
OUTPUT = Path("/kaggle/working/beam-results")
OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
from pathlib import Path
import subprocess,sys
script=Path('/tmp/validate_graphs.py')
script.write_text('import json,time,traceback,hashlib,itertools\nfrom pathlib import Path\nfrom collections import deque\nimport numpy as np\nimport torch,cayleypy,cayleypy_native\nfrom cayleypy import CayleyGraph,CayleyGraphDef,PermutationGroups as P,Puzzles,MatrixGroups\nfrom cayleypy_native import NativeOptions,NativeUnavailable,enable_native,disable_native\nfrom cayleypy_native.contracts import GraphContract\nfrom cayleypy.models.checkpoint import graph_hash\nOUT=Path(\'/kaggle/working/graphs-results\');OUT.mkdir(exist_ok=True)\nCACHE=Path(\'/tmp/cayleypy-native-graphs-cache\')\nrecords=[];contracts=[]\ndef save():\n (OUT/\'results.json\').write_text(json.dumps({\'torch\':torch.__version__,\'cayleypy\':getattr(cayleypy,\'__version__\',\'unknown\'),\'module\':cayleypy_native.__file__,\'contracts\':contracts,\'records\':records},indent=2,default=str))\ndef gather(state,g):return tuple(state[i] for i in g)\ndef sample_start(definition):\n center=tuple(definition.central_state);gens=[tuple(g) for g in definition.generators_permutations]\n inverse=[]\n for g in gens:\n  inv=[0]*len(g)\n  for i,j in enumerate(g):inv[j]=i\n  inverse.append(tuple(inv))\n seen={center};q=deque([(center,0)])\n while q:\n  state,d=q.popleft()\n  if d==2:return state,d\n  for g in inverse:\n   s=gather(state,g)\n   if s not in seen:seen.add(s);q.append((s,d+1))\n raise AssertionError(\'no two-move nontrivial start\')\nbase=P.lrx(4)\ndef colored(center):return CayleyGraphDef.create(base.generators_permutations,central_state=center,generator_names=list(base.generator_names))\ncases=[(\'lrx4\',base),(\'coxeter4\',P.coxeter(4)),(\'pancake4\',P.pancake(4)),(\'top_spin4\',P.top_spin(4,3)),\n (\'colored_lrx4\',colored([0,0,1,1])),(\'nonidentity_lrx4\',colored([3,1,0,2])),(\'sparse_labels_lrx4\',colored([3,1,3,1])),\n (\'quoted_names\',CayleyGraphDef.create(base.generators_permutations,generator_names=[\'a.quote"\',\'b.newline\\n\',\'move.Юникод\'])),\n (\'cyclic_coxeter4\',P.cyclic_coxeter(4)),(\'all_transpositions4\',P.all_transpositions(4)),(\'full_reversals4\',P.full_reversals(4)),\n (\'burnt_pancake2\',P.burnt_pancake(2)),(\'directed_cycle4\',CayleyGraphDef.create([[1,2,3,0]],generator_names=[\'forward\'])),\n (\'pancake5\',P.pancake(5)),(\'lrx8\',P.lrx(8)),(\'rubik2_fixed_QTM\',Puzzles.rubik_cube(2,\'fixed_QTM\'))]\n# Inspect mathematical ingestion before any shape build.\nfor name,definition in cases:\n start,d=sample_start(definition);graph=CayleyGraph(definition,device=\'cuda\',num_gpus=1)\n c=GraphContract.from_graph(graph,np.array(start,dtype=np.int64));assert c.graph_hash==graph_hash(definition)\n assert c.center==tuple(definition.central_state) and c.generators==tuple(tuple(g) for g in definition.generators_permutations)\n for i,g in enumerate(c.generators):\n  expected=gather(start,g);actual=graph.apply_path(list(start),[i])[0].cpu().tolist();assert tuple(actual)==expected,(name,i)\n contracts.append({\'case\':name,\'state_len\':c.state_len,\'move_count\':c.move_count,\'classes\':c.num_classes,\'graph_hash\':c.graph_hash,\'start\':start,\'center\':c.center,\'generators\':c.generators,\'generator_names\':c.generator_names,\'known_distance\':d});save()\nprint(\'GRAPH_CONTRACTS_PASS\',len(contracts),flush=True)\nfor name,definition in cases:\n start,d=sample_start(definition)\n for world in (1,2):\n  graph=CayleyGraph(definition,device=\'cuda\',num_gpus=1)\n  enable_native(NativeOptions(num_gpus=world,cache_dir=CACHE,timeout_seconds=120,build_timeout_seconds=1800))\n  try:\n   before=time.monotonic();result=graph.beam_search(start_state=list(start),beam_width=128,max_steps=8,return_path=True,backend=\'native\')\n   assert result.backend==\'native\' and result.path_found and result.path_length>=d\n   c=GraphContract.from_graph(graph,list(start));assert c.replay(result.path)\n   assert tuple(graph.apply_path(list(start),result.path)[0].cpu().tolist())==c.center\n   assert result.native_metadata[\'devices\']==list(range(world))\n   assert result.native_metadata[\'inference_backend\']==(\'libtorch\' if \'T4\' in torch.cuda.get_device_name(0).split() else \'cutlass\')\n   if world>1:assert result.native_metadata[\'worker_logs_complete\'] and len(result.native_metadata[\'worker_streams\'])==world*2\n   records.append({\'case\':name,\'world\':world,\'executor\':result.native_metadata[\'inference_backend\'],\'status\':\'PASS\',\'path\':result.path,\'path_string\':result.get_path_as_string(),\'elapsed\':time.monotonic()-before,\'metadata\':result.native_metadata});save();print(\'GPU_CASE_PASS\',name,world,flush=True)\n  except Exception:\n   records.append({\'case\':name,\'world\':world,\'status\':\'FAIL\',\'error\':traceback.format_exc()});save();raise\n  finally:disable_native()\n# Same-shape cache reuse must not reuse the wrong graph target or generator order.\nfor name,definition in cases:\n if name not in (\'colored_lrx4\',\'nonidentity_lrx4\',\'sparse_labels_lrx4\'):continue\n start,d=sample_start(definition)\n for world in (1,2):\n  graph=CayleyGraph(definition,device=\'cuda\',num_gpus=1);enable_native(NativeOptions(num_gpus=world,cache_dir=CACHE,inference_backend=\'libtorch\',timeout_seconds=120))\n  try:\n   result=graph.beam_search(start_state=list(start),beam_width=128,max_steps=8,return_path=True)\n   assert result.path_found and GraphContract.from_graph(graph,list(start)).replay(result.path)\n   assert result.native_metadata[\'inference_backend\']==\'libtorch\'\n   records.append({\'case\':name,\'world\':world,\'executor\':\'libtorch\',\'status\':\'PASS\',\'path\':result.path,\'metadata\':result.native_metadata});save();print(\'LIBTORCH_CASE_PASS\',name,world,flush=True)\n  finally:disable_native()\n# Unsupported graphs and unreachable states are distinct outcomes.\nfor name,definition in [(\'matrix\',MatrixGroups.heisenberg()),(\'oversize\',P.lrx(121))]:\n graph=CayleyGraph(definition,device=\'cuda\',num_gpus=1);enable_native(NativeOptions(num_gpus=2,cache_dir=CACHE))\n try:\n  try:graph.beam_search(start_state=graph.central_state,beam_width=128,max_steps=8)\n  except NativeUnavailable as e:records.append({\'case\':name,\'status\':\'REJECTED_EXPECTED\',\'error\':str(e)});save()\n  else:raise AssertionError(\'unsupported input accepted\')\n finally:disable_native()\nprint(\'ALL_GRAPH_CASES_PASS\',len(records),flush=True)\n\nfor name,definition in cases:\n graph=CayleyGraph(definition,device=\'cuda\',num_gpus=1);start,d=sample_start(definition)\n enable_native(NativeOptions(num_gpus=2,cache_dir=CACHE))\n try:\n  r=graph.beam_search(start_state=list(start),beam_width=128,max_steps=8,return_path=True,backend=\'torch\')\n  assert r.path_found and GraphContract.from_graph(graph,list(start)).replay(r.path)\n  records.append({\'case\':name,\'executor\':\'cayleypy\',\'status\':\'PASS\',\'path\':r.path});save();print(\'CAYLEYPY_CASE_PASS\',name,flush=True)\n finally:disable_native()\n# Same multiset is insufficient for reachability: this start is outside C4 orbit.\ndefinition=CayleyGraphDef.create([[1,2,3,0]],generator_names=[\'forward\'])\nfor world in (1,2):\n graph=CayleyGraph(definition,device=\'cuda\',num_gpus=1);enable_native(NativeOptions(num_gpus=world,cache_dir=CACHE,timeout_seconds=120))\n try:\n  r=graph.beam_search(start_state=[0,1,3,2],beam_width=128,max_steps=4,return_path=True)\n  assert not r.path_found and r.path is None\n  records.append({\'case\':\'directed_unreachable\',\'world\':world,\'status\':\'NOT_FOUND_EXPECTED\',\'metadata\':r.native_metadata});save();print(\'UNREACHABLE_CASE_PASS\',world,flush=True)\n finally:disable_native()\nprint(\'ALL_EXTRA_GRAPH_CASES_PASS\',len(records),flush=True)\nsave()\n',encoding='utf8')
try:
    subprocess.run([sys.executable,str(script)],check=True)
finally:
    import shutil
    cache=Path('/tmp/cayleypy-native-graphs-cache')
    dest=Path('/kaggle/working/graphs-diagnostics')
    for folder in ('runs','builds'):
        for f in (cache/folder).rglob('*'):
            if f.is_file() and f.suffix in {'.log','.json','.csv','.fp16'} and f.stat().st_size<20_000_000:
                target=dest/f.relative_to(cache)
                target.parent.mkdir(parents=True,exist_ok=True)
                shutil.copy2(f,target)

